In [1]:
# Data Inspection, Data Cleaning, Feature Engineering, and Exploratory Data Analysis (EDA)
# ----------------------------------------------------------------------------
# This section covers the following steps:
# 1. **Data Inspection**
#    - Checking shape
#    - Checking columns
#    - Checking for missing values
#    - Checking for duplicates
# 2. **Data Cleaning**: 
#    - Handling missing values, duplicates, and converting data types.
#    - Ensuring the dataset is ready for analysis.
# 3. **Feature Engineering**: 
#    - Creating new features such as 'Age Group', 'Customer Total Spend', 'Revenue', etc.
#    - Transforming and enhancing the data for better insights.
# 4. **Exploratory Data Analysis (EDA)**: 
#    - Calculating key metrics and visualizing the patterns in the dataset.
#    - Gaining insights into trends and relationships within the data.

In [36]:
# Import necessary libraries
import pandas as pd

# Load the dataset
file_path = r'C:\Users\Abdulsalam\Downloads\payment_logs_23_24.csv' # Adjust the path if needed
data = pd.read_csv(file_path)

# 1️⃣ Data Inspection
# Check shape (number of rows and columns)
print("----------------------------------")
print("Data Inspection Result")
print(f"Dataset Shape: {data.shape}")

# Check columns
print(f"Columns in the dataset: {data.columns}")

# Check for missing values
missing_values = data.isnull().sum()
print(f"Missing values:\n{missing_values}")

# Check for duplicates
duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Check data types
data_types = data.dtypes
print(f"Data Types:\n{data_types}")

# 2️⃣ Data Cleaning
# Convert 'Purchase Date' column to datetime
data['Purchase Date'] = pd.to_datetime(data['Purchase Date'], errors='coerce')

# Handle missing values
# Fill missing values in the 'Gender' column with the mode (most frequent value)
data['Gender'].fillna(data['Gender'].mode()[0], inplace=True)

# Remove duplicate rows if any
data.drop_duplicates(inplace=True)

# Check again for missing values after cleaning
missing_values_cleaned = data.isnull().sum()
print("----------------------------------")
print("Data Cleaniing Result : ")
print(f"Missing values after cleaning:\n{missing_values_cleaned}")

# 3️⃣ Feature Engineering
# Create 'Month', 'Quarter', 'Year' columns from 'Purchase Date'
data['Month'] = data['Purchase Date'].dt.month
data['Quarter'] = data['Purchase Date'].dt.quarter
data['Year'] = data['Purchase Date'].dt.year

# Create 'Revenue' column (if not already present)
if 'Total Price' in data.columns:
    # If 'Add-on Total' exists, include it in the revenue calculation
    if 'Add-on Total' in data.columns:
        data['Revenue'] = data['Total Price'] + data['Add-on Total']
    else:
        # Fallback to Total Price if Add-on Total is missing
        data['Revenue'] = data['Total Price']
else:
    # Fallback to Quantity * Unit Price if Total Price is not present
    data['Revenue'] = data['Quantity'] * data['Unit Price']

# Age Grouping
# Define bins and labels for age groups
bins = [0, 25, 35, 45, 60, 100]  # Define age bins
labels = ['18-25', '26-35', '36-45', '46-60', '60+']  # Age group labels
data['Age Group'] = pd.cut(data['Age'], bins=bins, labels=labels, right=False)  # Create Age Group column

# Customer Total Spend (sum of all revenue per customer)
customer_spend = data.groupby('Customer ID')['Revenue'].sum().reset_index()
customer_spend.rename(columns={'Revenue': 'Customer Total Spend'}, inplace=True)

# Customer Purchase Count (number of purchases per customer)
customer_count = data.groupby('Customer ID')['Order Status'].count().reset_index()
customer_count.rename(columns={'Order Status': 'Customer Purchase Count'}, inplace=True)

# Merge these new columns with the original dataset
data = data.merge(customer_spend, on='Customer ID', how='left')
data = data.merge(customer_count, on='Customer ID', how='left')

# Final cleaned dataset overview
print("----------------------------------")
print("New_Rows_Added_Dataset : ")
print(f"Cleaned Data Sample:\n{data.head()}")
# Save the cleaned and updated dataset to a new CSV file
output_file_path = r'C:\Users\Abdulsalam\Downloads\cleaned_payment_logs_23_24.csv'  # Adjust the path as needed
data.to_csv(output_file_path, index=False)

print(f"Cleaned dataset has been saved to {output_file_path}")

----------------------------------
Data Inspection Result
Dataset Shape: (20000, 16)
Columns in the dataset: Index(['Customer ID', 'Age', 'Gender', 'Loyalty Member', 'Product Type', 'SKU',
       'Rating', 'Order Status', 'Payment Method', 'Total Price', 'Unit Price',
       'Quantity', 'Purchase Date', 'Shipping Type', 'Add-ons Purchased',
       'Add-on Total'],
      dtype='object')
Missing values:
Customer ID             0
Age                     0
Gender                  1
Loyalty Member          0
Product Type            0
SKU                     0
Rating                  0
Order Status            0
Payment Method          0
Total Price             0
Unit Price              0
Quantity                0
Purchase Date           0
Shipping Type           0
Add-ons Purchased    4868
Add-on Total            0
dtype: int64
Number of duplicate rows: 0
Data Types:
Customer ID            int64
Age                    int64
Gender                object
Loyalty Member        object
Product Ty

C:\Users\Abdulsalam\AppData\Local\Temp\ipykernel_32796\3450953052.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Gender'].fillna(data['Gender'].mode()[0], inplace=True)


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\Abdulsalam\\Downloads\\cleaned_payment_logs_23_24.csv'

In [16]:
import pandas as pd

# Load the cleaned dataset
file_path = r'C:\Users\Abdulsalam\Downloads\cleaned_payment_logs_23_24.csv'  # Adjust the path if needed
data = pd.read_csv(file_path)

# Exclude cancelled orders
data = data[data['Order Status'] != 'Cancelled']

# 1. Total Revenue (excluding cancelled orders)
total_revenue = data['Revenue'].sum()

# 2. Total Orders (excluding cancelled orders)
total_orders = data.shape[0]

# 3. Total Customers (excluding cancelled orders)
total_customers = data['Customer ID'].nunique()

# 4. Average Order Value (AOV) (excluding cancelled orders)
aov = total_revenue / total_orders

# 5. Revenue by Month
revenue_by_month = data.groupby('Month')['Revenue'].sum()

# 6. Revenue by Category
revenue_by_category = data.groupby('Product Type')['Revenue'].sum()

# 7. Revenue by Gender
revenue_by_gender = data.groupby('Gender')['Revenue'].sum()

# 8. Revenue by Payment Method
revenue_by_payment_method = data.groupby('Payment Method')['Revenue'].sum()

print("-------------------------")
print("EXPLORATORY DATA ANALYSIS RESULT :")
print("")

# Display KPIs
print(f"Total Revenue (Excluding Cancelled Orders): ${total_revenue:,.2f}")
print(f"Total Orders (Excluding Cancelled): {total_orders}")
print(f"Total Customers (Excluding Cancelled): {total_customers}")
print(f"Average Order Value (AOV) (Excluding Cancelled): ${aov:,.2f}")

# Display revenue breakdowns
print("\nRevenue by Month:")
print(revenue_by_month)

print("\nRevenue by Category:")
print(revenue_by_category)

print("\nRevenue by Gender:")
print(revenue_by_gender)

print("\nRevenue by Payment Method:")
print(revenue_by_payment_method)

-------------------------
EXPLORATORY DATA ANALYSIS RESULT :

Total Revenue (Excluding Cancelled Orders): $42,629,615.57
Total Orders (Excluding Cancelled): 13432
Total Customers (Excluding Cancelled): 9466
Average Order Value (AOV) (Excluding Cancelled): $3,173.74

Revenue by Month:
Month
1     4516277.81
2     3897250.16
3     4226126.65
4     4300323.69
5     4462915.20
6     4465994.71
7     4460946.48
8     4378433.10
9     3665882.54
10    1555325.86
11    1390116.50
12    1310022.87
Name: Revenue, dtype: float64

Revenue by Category:
Product Type
Headphones     2734651.00
Laptop         8365905.25
Smartphone    14407835.84
Smartwatch     9398591.23
Tablet         7722632.25
Name: Revenue, dtype: float64

Revenue by Gender:
Gender
Female    21256916.03
Male      21372699.54
Name: Revenue, dtype: float64

Revenue by Payment Method:
Payment Method
Bank Transfer     8450693.44
Cash              4392521.96
Credit Card      12579688.01
Debit Card        4545850.34
PayPal            85

In [35]:
import pandas as pd

# Load the cleaned dataset
file_path = r'C:\Users\Abdulsalam\Downloads\cleaned_payment_logs_23_24.csv'  # Adjust the path if needed
data = pd.read_csv(file_path)

# Ensure 'Purchase Date' is in datetime format (in case it's not already)
data['Purchase Date'] = pd.to_datetime(data['Purchase Date'], errors='coerce')

# Extract Year and Quarter from Purchase Date
data['Year'] = data['Purchase Date'].dt.year
data['Quarter'] = data['Purchase Date'].dt.quarter

# 1️⃣ Customer Segmentation (Top 20% and Bottom 20% customers)
# Calculate Total Spend per Customer
customer_total_spend = data.groupby('Customer ID')['Revenue'].sum().reset_index()
customer_total_spend.rename(columns={'Revenue': 'Customer Total Spend'}, inplace=True)

# Calculate Purchase Frequency per Customer
customer_purchase_freq = data.groupby('Customer ID')['Order Status'].count().reset_index()
customer_purchase_freq.rename(columns={'Order Status': 'Customer Purchase Frequency'}, inplace=True)

# Merge both metrics: Total Spend and Purchase Frequency
customer_segmentation = customer_total_spend.merge(customer_purchase_freq, on='Customer ID', how='left')

# Top 20% customers by Total Spend
top_20_percent_threshold = customer_segmentation['Customer Total Spend'].quantile(0.8)
top_20_percent_customers = customer_segmentation[customer_segmentation['Customer Total Spend'] >= top_20_percent_threshold]

# Low Spenders (Bottom 20%) based on Total Spend
bottom_20_percent_threshold = customer_segmentation['Customer Total Spend'].quantile(0.2)
low_spenders = customer_segmentation[customer_segmentation['Customer Total Spend'] <= bottom_20_percent_threshold]

# 2️⃣ Identifying Specific Customer Segments:
# 1. High spenders with low frequency (Top 20% in spend, Bottom 50% in frequency)
high_spenders_low_freq_threshold = customer_segmentation['Customer Purchase Frequency'].quantile(0.5)  # Bottom 50% in frequency
high_spenders_low_freq = customer_segmentation[(customer_segmentation['Customer Total Spend'] >= top_20_percent_threshold) &
                                                (customer_segmentation['Customer Purchase Frequency'] <= high_spenders_low_freq_threshold)]

# 2. Frequent buyers with low spend (Bottom 50% in spend, Top 20% in frequency)
frequent_buyers_low_spend_threshold = customer_segmentation['Customer Total Spend'].quantile(0.5)  # Bottom 50% in spend
frequent_buyers_low_spend = customer_segmentation[(customer_segmentation['Customer Total Spend'] <= frequent_buyers_low_spend_threshold) &
                                                  (customer_segmentation['Customer Purchase Frequency'] >= top_20_percent_threshold)]

# 3️⃣ Behavioral Analysis Results - Print Outputs
print("Behavioral Analysis : ")
print(f"Top 20% customers: {top_20_percent_customers.shape[0]} customers")
print(f"Low spenders: {low_spenders.shape[0]} customers")
print(f"High spenders with low frequency: {high_spenders_low_freq.shape[0]} customers")
print(f"Frequent buyers with low spend: {frequent_buyers_low_spend.shape[0]} customers")

# 4️⃣ Repeat vs New Customers
# Repeat Customers: those who made more than 1 purchase
repeat_customers = data.groupby('Customer ID')['Order Status'].count().reset_index()
repeat_customers = repeat_customers[repeat_customers['Order Status'] > 1]

# Calculate percentage of Repeat Customers
repeat_percentage = (repeat_customers.shape[0] / data['Customer ID'].nunique()) * 100

# Calculate Average Purchase Frequency of Repeat Customers
avg_purchase_freq = repeat_customers['Order Status'].mean()

print(f"Percentage of Repeat Customers: {repeat_percentage:.2f}%")
print(f"Average Purchase Frequency of Repeat Customers: {avg_purchase_freq:.2f}")

# 5️⃣ Trend Analysis
# Calculate Monthly Revenue Growth Percentage (Month-over-Month change)
monthly_revenue_growth = data.groupby('Month')['Revenue'].sum().pct_change() * 100

# Identify Peak Month with Maximum Revenue
revenue_by_month = data.groupby('Month')['Revenue'].sum()
peak_month = revenue_by_month.idxmax()  # Month with highest revenue
peak_month_revenue = revenue_by_month.max()

print(f"Peak month: {peak_month} with revenue of ${peak_month_revenue:,.2f}")

# Identify Seasonal Spikes (if any)
seasonal_spikes = monthly_revenue_growth[monthly_revenue_growth > 10]  # Growth > 10%
print(f"Seasonal spikes in months: {seasonal_spikes.index.tolist()}")

# 6️⃣ **Additional Analysis**:
# 1. Which Quarter Produced the Most Revenue for Each Year
revenue_by_quarter = data.groupby(['Year', 'Quarter'])['Revenue'].sum().reset_index()

# Identify the quarter with the most revenue for each year
max_revenue_quarter = revenue_by_quarter.loc[revenue_by_quarter.groupby('Year')['Revenue'].idxmax()]

# 2. Which Age Group Produced More Revenue for Each Year
revenue_by_age_group = data.groupby(['Year', 'Age Group'])['Revenue'].sum().reset_index()

# Identify the age group with the most revenue for each year
max_revenue_age_group = revenue_by_age_group.loc[revenue_by_age_group.groupby('Year')['Revenue'].idxmax()]

# 3. Most Revenue Generating Age Group in Each Quarter of Each Year
revenue_by_age_group_quarter = data.groupby(['Year', 'Quarter', 'Age Group'])['Revenue'].sum().reset_index()
max_revenue_quarter = max_revenue_quarter.reset_index(drop=True)
max_revenue_age_group = max_revenue_age_group.reset_index(drop=True)
max_revenue_age_group_quarter = max_revenue_age_group_quarter.reset_index(drop=True)

# **Print results for the new analysis**:

print("\nQuarter with the most revenue for each year:")
print(max_revenue_quarter)

print("\nAge group with the most revenue for each year:")
print(max_revenue_age_group)

Behavioral Analysis : 
Top 20% customers: 2430 customers
Low spenders: 2457 customers
High spenders with low frequency: 228 customers
Frequent buyers with low spend: 0 customers
Percentage of Repeat Customers: 45.31%
Average Purchase Frequency of Repeat Customers: 2.43
Peak month: 5 with revenue of $6,709,042.93
Seasonal spikes in months: [3]

Quarter with the most revenue for each year:
   Year  Quarter      Revenue
0  2023        4   6367600.82
1  2024        2  19795930.14

Age group with the most revenue for each year:
   Year Age Group      Revenue
0  2023       60+   2289803.95
1  2024       60+  18837883.16
